In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')


In [2]:
Data_path='../data/amex_selected_features.parquet'

df=pd.read_parquet(Data_path)
print(f'Shape: {df.shape}')
print(f'Features: {df.shape[1] - 1}')
print(f'Target distribution:')
print(df['target'].value_counts())
print(f'Default rate: {df["target"].mean():.4f}')

Shape: (458913, 130)
Features: 129
Target distribution:
target
0    340085
1    118828
Name: count, dtype: int64
Default rate: 0.2589


In [3]:
cat_cols = [col for col in ['B_30', 'B_38', 'D_114', 'D_116', 'D_117',
                              'D_120', 'D_126', 'D_63', 'D_64', 'D_66', 'D_68']
            if col in df.columns]

label_encoders={}

for col in cat_cols:
    le=LabelEncoder()
    df[col]=le.fit_transform(df[col].astype(str))
    label_encoders[col]=le

print(f'Encoded {len(cat_cols)} categorical columns')
print(f'Categorical columns present : {cat_cols}')

Encoded 8 categorical columns
Categorical columns present : ['B_30', 'B_38', 'D_114', 'D_117', 'D_120', 'D_63', 'D_64', 'D_68']


In [4]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'Train default rate: {y_train.mean():.4f}')
print(f'Test default rate: {y_test.mean():.4f}')

Training set: (367130, 129)
Test set: (91783, 129)
Train default rate: 0.2589
Test default rate: 0.2589


In [5]:
def evaluate_model(model_name, y_true, y_pred_proba):
    
    auc_roc = roc_auc_score(y_true, y_pred_proba)
    
    gini = 2 * auc_roc - 1
    
    # KS Statistic
    # sort customers by predicted probability descending
    # find the maximum difference between cumulative good and bad rates
    df_eval = pd.DataFrame({'y_true': y_true, 'y_proba': y_pred_proba})
    df_eval = df_eval.sort_values('y_proba', ascending=False).reset_index(drop=True)
    
    total_bad = df_eval['y_true'].sum()
    total_good = (df_eval['y_true'] == 0).sum()
    
    df_eval['cum_bad_rate'] = df_eval['y_true'].cumsum() / total_bad
    df_eval['cum_good_rate'] = (df_eval['y_true'] == 0).cumsum() / total_good
    
    ks_stat = (df_eval['cum_bad_rate'] - df_eval['cum_good_rate']).abs().max()
    
    print(f'{model_name}:')
    print(f'  AUC-ROC:  {auc_roc:.4f}')
    print(f'  Gini:     {gini:.4f}')
    print(f'  KS Stat:  {ks_stat:.4f}')
    print()
    
    return {'model': model_name, 'auc_roc': auc_roc, 'gini': gini, 'ks_stat': ks_stat}

In [6]:
# convert to float32 to halve SMOTE memory requirement
X_train_f32 = X_train.astype('float32')

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_f32, y_train)

# convert back to float32 after SMOTE (it outputs float64)
X_train_smote = X_train_smote.astype('float32')

# free the temporary float32 copy
del X_train_f32
import gc
gc.collect()

print(f'Original training set shape: {X_train.shape}')
print(f'SMOTE resampled shape: {X_train_smote.shape}')
print(f'Original default rate: {y_train.mean():.4f}')
print(f'SMOTE default rate: {y_train_smote.mean():.4f}')

Original training set shape: (367130, 129)
SMOTE resampled shape: (544136, 129)
Original default rate: 0.2589
SMOTE default rate: 0.5000


In [7]:
import gc
results = []

# ------- Logistic Regression -------
print('LOGISTIC REGRESSION')

lr_none = LogisticRegression(max_iter=1000, random_state=42)
lr_none.fit(X_train, y_train)
results.append(evaluate_model('LR — No Handling', y_test, lr_none.predict_proba(X_test)[:, 1]))

lr_cw = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_cw.fit(X_train, y_train)
results.append(evaluate_model('LR — Class Weights', y_test, lr_cw.predict_proba(X_test)[:, 1]))

lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)
results.append(evaluate_model('LR — SMOTE', y_test, lr_smote.predict_proba(X_test)[:, 1]))

del lr_none, lr_smote
gc.collect()

# ------- Random Forest ------
print('RANDOM FOREST')

rf_none = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_none.fit(X_train, y_train)
results.append(evaluate_model('RF — No Handling', y_test, rf_none.predict_proba(X_test)[:, 1]))

rf_cw = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
rf_cw.fit(X_train, y_train)
results.append(evaluate_model('RF — Class Weights', y_test, rf_cw.predict_proba(X_test)[:, 1]))

rf_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)
results.append(evaluate_model('RF — SMOTE', y_test, rf_smote.predict_proba(X_test)[:, 1]))

del rf_none, rf_cw, rf_smote
gc.collect()

# ------- XGBoost -------

print('XGBOOST')


scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_none = xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                               eval_metric='auc', tree_method='hist', max_depth=4)
xgb_none.fit(X_train, y_train)
results.append(evaluate_model('XGB — No Handling', y_test, xgb_none.predict_proba(X_test)[:, 1]))

xgb_cw = xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                              eval_metric='auc', tree_method='hist', max_depth=4,
                              scale_pos_weight=scale_pos_weight)
xgb_cw.fit(X_train, y_train)
results.append(evaluate_model('XGB — Class Weights', y_test, xgb_cw.predict_proba(X_test)[:, 1]))

xgb_smote = xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                                eval_metric='auc', tree_method='hist', max_depth=4)
xgb_smote.fit(X_train_smote, y_train_smote)
results.append(evaluate_model('XGB — SMOTE', y_test, xgb_smote.predict_proba(X_test)[:, 1]))

del xgb_none, xgb_cw, xgb_smote
gc.collect()

# ------- LightGBM ------
print('LIGHTGBM')


lgb_none = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
lgb_none.fit(X_train, y_train)
results.append(evaluate_model('LGB — No Handling', y_test, lgb_none.predict_proba(X_test)[:, 1]))

lgb_cw = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                              class_weight='balanced', verbose=-1)
lgb_cw.fit(X_train, y_train)
results.append(evaluate_model('LGB — Class Weights', y_test, lgb_cw.predict_proba(X_test)[:, 1]))

lgb_smote = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
lgb_smote.fit(X_train_smote, y_train_smote)
results.append(evaluate_model('LGB — SMOTE', y_test, lgb_smote.predict_proba(X_test)[:, 1]))

print('All models trained and evaluated')

LOGISTIC REGRESSION
LR — No Handling:
  AUC-ROC:  0.9566
  Gini:     0.9132
  KS Stat:  0.7806

LR — Class Weights:
  AUC-ROC:  0.9566
  Gini:     0.9132
  KS Stat:  0.7808

LR — SMOTE:
  AUC-ROC:  0.9559
  Gini:     0.9118
  KS Stat:  0.7793

RANDOM FOREST
RF — No Handling:
  AUC-ROC:  0.9546
  Gini:     0.9092
  KS Stat:  0.7784

RF — Class Weights:
  AUC-ROC:  0.9534
  Gini:     0.9068
  KS Stat:  0.7772

RF — SMOTE:
  AUC-ROC:  0.9523
  Gini:     0.9045
  KS Stat:  0.7769

XGBOOST
XGB — No Handling:
  AUC-ROC:  0.9588
  Gini:     0.9175
  KS Stat:  0.7882

XGB — Class Weights:
  AUC-ROC:  0.9587
  Gini:     0.9173
  KS Stat:  0.7865

XGB — SMOTE:
  AUC-ROC:  0.9569
  Gini:     0.9139
  KS Stat:  0.7824

LIGHTGBM
LGB — No Handling:
  AUC-ROC:  0.9590
  Gini:     0.9179
  KS Stat:  0.7886

LGB — Class Weights:
  AUC-ROC:  0.9587
  Gini:     0.9174
  KS Stat:  0.7865

LGB — SMOTE:
  AUC-ROC:  0.9573
  Gini:     0.9145
  KS Stat:  0.7836

All models trained and evaluated


In [8]:
results_df = pd.DataFrame(results).sort_values('auc_roc', ascending=False).reset_index(drop=True)

print('Model Comparison — sorted by AUC-ROC:')
print(results_df.to_string(index=False))
print()
print(f'Best model: {results_df.iloc[0]["model"]}')
print(f'Best AUC-ROC: {results_df.iloc[0]["auc_roc"]:.4f}')
print(f'Best Gini: {results_df.iloc[0]["gini"]:.4f}')
print(f'Best KS Stat: {results_df.iloc[0]["ks_stat"]:.4f}')

Model Comparison — sorted by AUC-ROC:
              model  auc_roc     gini  ks_stat
  LGB — No Handling 0.958969 0.917938 0.788613
  XGB — No Handling 0.958756 0.917512 0.788153
LGB — Class Weights 0.958683 0.917367 0.786474
XGB — Class Weights 0.958650 0.917300 0.786496
        LGB — SMOTE 0.957259 0.914518 0.783631
        XGB — SMOTE 0.956929 0.913858 0.782381
   LR — No Handling 0.956585 0.913170 0.780608
 LR — Class Weights 0.956579 0.913159 0.780849
         LR — SMOTE 0.955906 0.911812 0.779267
   RF — No Handling 0.954577 0.909153 0.778403
 RF — Class Weights 0.953414 0.906828 0.777170
         RF — SMOTE 0.952268 0.904537 0.776852

Best model: LGB — No Handling
Best AUC-ROC: 0.9590
Best Gini: 0.9179
Best KS Stat: 0.7886


In [10]:
# best model — Logistic Regression Class Weights
# chosen over LightGBM (AUC 0.9590 vs 0.9566) because:
# LR produces log-odds natively — required for PDO scorecard scaling
# performance difference is negligible (0.0024 AUC)
# industry standard for scorecard conversion

best_model = lr_cw

MODEL_PATH    = r'..\models\best_model.pkl'
FEATURES_PATH = r'..\models\model_columns.pkl'
ENCODERS_PATH = r'..\models\label_encoders.pkl'

joblib.dump(best_model, MODEL_PATH)
joblib.dump(list(X_train.columns), FEATURES_PATH)
joblib.dump(label_encoders, ENCODERS_PATH)

print(f'Model saved: Logistic Regression — Class Weights')
print(f'AUC-ROC: 0.9566')
print(f'Gini: 0.9132')
print(f'KS Stat: 0.7808')
print(f'Feature columns saved: {len(X_train.columns)}')
print(f'Label encoders saved: {len(label_encoders)}')

Model saved: Logistic Regression — Class Weights
AUC-ROC: 0.9566
Gini: 0.9132
KS Stat: 0.7808
Feature columns saved: 129
Label encoders saved: 8
